# Diabetic Retinopathy — Training Control Panel

This notebook is a **manual control panel**. It does not start training when opened.

- Dataset: local `data/btgraham-300/`, preserving the existing `train`, `validation`, and `test` splits.
- Common model input: `300 × 300` RGB; the shared loader performs the resize.
- Current experiment block: EfficientNetB0 baseline (frozen ImageNet backbone + five-class head).
- Recovery checkpoints and best-validation model are stored separately for each run.

**GPU setup:** the shared `configs/tensorflow_setup.py` registers the Conda environment DLL directory before TensorFlow is imported. The PyCharm GPU matrix-multiplication test succeeded on the RTX 3050. Restart this notebook kernel after replacing project files so the setup runs before TensorFlow loads.

## 1. Environment and device check
Run this cell before deciding whether to train. It does not initialize a model or start training.

In [1]:
from configs import tensorflow_setup as _tensorflow_setup

import sys
import numpy as np
import tensorflow as tf

_tensorflow_setup.configure_gpu_memory_growth(tf)

import tensorflow_datasets as tfds

print("Python:", sys.version.split()[0])
print("CUDA DLL directory:", _tensorflow_setup.CUDA_DLL_DIRECTORY)
print("TensorFlow:", tf.__version__)
print("Keras:", tf.keras.__version__ if hasattr(tf.keras, "__version__") else "not exposed")
print("NumPy:", np.__version__)
print("TFDS:", tfds.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))
print("EfficientNetB0 API:", hasattr(tf.keras.applications, "EfficientNetB0"))
print("Seed utility API:", hasattr(tf.keras.utils, "set_random_seed"))

Python: 3.10.20
CUDA DLL directory: C:\Users\kawal\miniconda3\envs\dr_tf\Library\bin
TensorFlow: 2.10.1
Keras: 2.10.0
NumPy: 1.23.5
TFDS: 4.8.3
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
EfficientNetB0 API: True
Seed utility API: True


## 2. Optional data-loader smoke test
This only reads one batch. It does not train or change the dataset.

In [2]:
from scripts.data_loader import inspect_dataset, make_datasets

inspect_dataset()
train_ds, validation_ds, test_ds = make_datasets(batch_size=4)
images, labels = next(iter(train_ds))

print("Image batch shape:", images.shape)
print("Image dtype:", images.dtype)
print("Pixel range:", float(tf.reduce_min(images)), "to", float(tf.reduce_max(images)))
print("Labels:", labels.numpy())

Dataset directory: D:\M L\Diabetic_Retinopathy_Detection_mini_project\data\btgraham-300
Dataset name:      diabetic_retinopathy_detection
Dataset version:   3.0.0
Features:          ['label', 'image', 'name']
Available splits:
  sample: 10 examples
  train: 35,126 examples
  validation: 10,906 examples
  test: 42,670 examples
Image feature:     Image(shape=(None, None, 3), dtype=uint8)
Label feature:     ClassLabel(shape=(), dtype=int64, num_classes=5)
Train     : train (35,126 examples)
Validation: validation (10,906 examples)
Test      : test (42,670 examples)
Image batch shape: (4, 300, 300, 3)
Image dtype: <dtype: 'float32'>
Pixel range: 0.0 to 255.00003051757812
Labels: [0 0 0 0]


## 3. EfficientNet — model initialization preview
This cell builds the architecture with `weights=None` only to inspect the model shape/parameter count. It does not load ImageNet weights or train. The training function separately creates a fresh model using the configured ImageNet weights.

In [3]:
from scripts.train_efficientnet import build_efficientnet_model

efficientnet_preview = build_efficientnet_model(weights=None)
efficientnet_preview.summary()
print("Input:", efficientnet_preview.input_shape)
print("Output:", efficientnet_preview.output_shape)
del efficientnet_preview
tf.keras.backend.clear_session()

Model: "efficientnet_b0_dr"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 retinal_image (InputLayer)  [(None, 300, 300, 3)]     0         
                                                                 
 efficientnetb0 (Functional)  (None, 10, 10, 1280)     4049571   
                                                                 
 global_average_pooling (Glo  (None, 1280)             0         
 balAveragePooling2D)                                            
                                                                 
 head_dropout (Dropout)      (None, 1280)              0         
                                                                 
 dr_severity (Dense)         (None, 5)                 6405      
                                                                 
Total params: 4,055,976
Trainable params: 6,405
Non-trainable params: 4,049,571
__________________________________

## 4. EfficientNet — manually controlled training
**Only run this cell when you are ready to train.** Choose `scratch` for a new isolated run or `resume` to continue the newest valid checkpoint. The epoch input is the **total target epoch**, not the number of additional epochs. If TensorFlow sees no GPU, CPU training requires an explicit opt-in.

In [5]:
from scripts.train_efficientnet import train_efficientnet

available_gpus = tf.config.list_physical_devices("GPU")
allow_cpu = False

if not available_gpus:
    print("WARNING: TensorFlow cannot currently see a GPU.")
    cpu_choice = input("To deliberately allow potentially slow CPU training, type RUN-ON-CPU: ").strip()
    allow_cpu = cpu_choice == "RUN-ON-CPU"

if not available_gpus and not allow_cpu:
    print("Stopped without training. Fix GPU visibility, or rerun and explicitly opt into CPU training.")
else:
    run_mode = input("Run mode [scratch/resume]: ").strip().lower()
    batch_text = input("Batch size [4]: ").strip()
    epochs_text = input("Total target epochs [10]: ").strip()

    batch_size = int(batch_text or 4)
    target_epochs = int(epochs_text or 10)

    efficientnet_result = train_efficientnet(
        run_mode=run_mode,
        batch_size=batch_size,
        epochs=target_epochs,
        allow_cpu=allow_cpu,
    )

Training device(s): /physical_device:GPU:0
Resuming run_004 from checkpoint: D:\M L\Diabetic_Retinopathy_Detection_mini_project\models\efficientnet\checkpoints\run_004\ckpt-1
Train     : train (35,126 examples)
Validation: validation (10,906 examples)
Test      : test (42,670 examples)
Input size: 300x300; classes: 5
Starting at epoch 1; target total epochs: 5
Epoch 2/5
2196/2196 [==============================] - 208s 93ms/step - loss: 0.7218 - accuracy: 0.7560 - val_loss: 0.7029 - val_accuracy: 0.7634
Epoch 3/5
2196/2196 [==============================] - 205s 93ms/step - loss: 0.7350 - accuracy: 0.7540 - val_loss: 0.6977 - val_accuracy: 0.7655
Epoch 4/5
2196/2196 [==============================] - 254s 116ms/step - loss: 0.7288 - accuracy: 0.7544 - val_loss: 0.6977 - val_accuracy: 0.7669
Epoch 5/5
2196/2196 [==============================] - 203s 92ms/step - loss: 0.7267 - accuracy: 0.7551 - val_loss: 0.6905 - val_accuracy: 0.7657
Run status: completed (5/5 epochs)
Recovery checkpoi

## 5. EfficientNet — inspect run status
This cell lists run metadata and checkpoint/model presence. It does not load or evaluate the model.

In [6]:
from scripts.utils import print_run_status

print_run_status("efficientnet")

Model: efficientnet
run_001: status=failed, epoch=0/10, checkpoint=no, best_model=no
  Output: D:\M L\Diabetic_Retinopathy_Detection_mini_project\models\efficientnet\runs\run_001
run_002: status=failed, epoch=0/5, checkpoint=no, best_model=no
  Output: D:\M L\Diabetic_Retinopathy_Detection_mini_project\models\efficientnet\runs\run_002
run_003: status=failed, epoch=1/5, checkpoint=yes, best_model=no
  Output: D:\M L\Diabetic_Retinopathy_Detection_mini_project\models\efficientnet\runs\run_003
run_004: status=completed, epoch=5/5, checkpoint=yes, best_model=yes
  Output: D:\M L\Diabetic_Retinopathy_Detection_mini_project\models\efficientnet\runs\run_004


[{'run_id': 'run_001',
  'status': 'failed',
  'last_completed_epoch': 0,
  'target_epochs': 10,
  'checkpoint': 'no',
  'best_model': 'no',
  'output_dir': 'D:\\M L\\Diabetic_Retinopathy_Detection_mini_project\\models\\efficientnet\\runs\\run_001'},
 {'run_id': 'run_002',
  'status': 'failed',
  'last_completed_epoch': 0,
  'target_epochs': 5,
  'checkpoint': 'no',
  'best_model': 'no',
  'output_dir': 'D:\\M L\\Diabetic_Retinopathy_Detection_mini_project\\models\\efficientnet\\runs\\run_002'},
 {'run_id': 'run_003',
  'status': 'failed',
  'last_completed_epoch': 1,
  'target_epochs': 5,
  'checkpoint': 'yes',
  'best_model': 'no',
  'output_dir': 'D:\\M L\\Diabetic_Retinopathy_Detection_mini_project\\models\\efficientnet\\runs\\run_003'},
 {'run_id': 'run_004',
  'status': 'completed',
  'last_completed_epoch': 5,
  'target_epochs': 5,
  'checkpoint': 'yes',
  'best_model': 'yes',
  'output_dir': 'D:\\M L\\Diabetic_Retinopathy_Detection_mini_project\\models\\efficientnet\\runs\\run_

## 6. DenseNet — model initialization / training / status (pending)
The DenseNet builder and training controls are not implemented yet. This block is intentionally non-executable for training: when added, it will have its own scratch/resume choice, batch-size prompt, target-epoch prompt, run-specific checkpoints, and status display. It will use the same `btgraham-300` split membership and 300 × 300 input.

## 7. EfficientNet + DenseNet fusion — model initialization / training / status (pending)
The fusion architecture is still a research decision and has not been implemented. This block will remain inactive until the standalone baselines are implemented and their resource use is understood. It will have independent checkpoints and run metadata.

## 8. EfficientNet — evaluation (pending)
Formal manual evaluation will be added in `scripts/evaluate.py` and will require explicit model/run selection. The training script does **not** automatically evaluate the test split. Keep the test split held out until the evaluation step.

## 9. Active model / interface activation (pending)
Training a model does not automatically make it the active model for inference. Activation will be implemented after evaluation and will explicitly record the chosen run and matching preprocessing/class order in `models/active/model_metadata.json`.